# 🕵️ Ontology Detective — Dashboard

> Build stamp: **2026-06-05 16:54:59**

Reads `DetectiveEvents` from **Datapolis_DetectiveEH** and shows cases solved,
accuracy, and detective rank across all sessions.

In [ ]:
import os, requests
from IPython.display import Markdown, display

EH_NAME = "Datapolis_DetectiveEH"
DB_NAME = "Datapolis_DetectiveEH"
try:
    import notebookutils
    WORKSPACE_ID = notebookutils.runtime.context.get("currentWorkspaceId")
    _gettoken    = notebookutils.credentials.getToken
except Exception:
    import mssparkutils
    WORKSPACE_ID = mssparkutils.runtime.context.get("currentWorkspaceId")
    _gettoken    = mssparkutils.credentials.getToken
FAB = "https://api.fabric.microsoft.com/v1"
def _fab(url):
    r = requests.get(url, headers={"Authorization": f"Bearer {_gettoken('pbi')}"}, timeout=60)
    r.raise_for_status(); return r.json()
dbs = _fab(f"{FAB}/workspaces/{WORKSPACE_ID}/items?type=KQLDatabase").get("value", [])
DB_ID = next(d for d in dbs if d["displayName"] == DB_NAME)["id"]
KQL_URI = _fab(f"{FAB}/workspaces/{WORKSPACE_ID}/kqlDatabases/{DB_ID}")["properties"]["queryServiceUri"]

def query_kql(csl):
    tok = _gettoken("kusto")
    r = requests.post(f"{KQL_URI}/v2/rest/query",
                      headers={"Authorization": f"Bearer {tok}", "Content-Type": "application/json"},
                      json={"csl": csl, "db": DB_NAME}, timeout=120)
    r.raise_for_status()
    for f in r.json():
        if f.get("FrameType") == "DataTable" and f.get("TableKind") == "PrimaryResult":
            cols = [c["ColumnName"] for c in f["Columns"]]
            return [dict(zip(cols, row)) for row in f["Rows"]]
    return []

## Cases solved per player

In [ ]:
rows = query_kql('''
DetectiveEvents
| where EventType in ('CaseSolved','WrongAccusation','BadgeIssued')
| summarize Solved=dcountif(CaseId, EventType=='CaseSolved'),
            Wrong=countif(EventType=='WrongAccusation'),
            Badges=countif(EventType=='BadgeIssued')
  by PlayerId
| order by Solved desc, Wrong asc
''')
if not rows:
    display(Markdown("_No detective activity yet._"))
else:
    md = ["| Player | Solved | Wrong calls | Badges |", "|--------|-------:|------------:|-------:|"]
    for r in rows:
        md.append(f"| `{r['PlayerId']}` | {r['Solved']} | {r['Wrong']} | {r['Badges']} |")
    display(Markdown("\n".join(md)))

## Recent events (last 25)

In [ ]:
rows = query_kql('''
DetectiveEvents
| order by Timestamp desc
| take 25
| project Timestamp, PlayerId, EventType, CaseId, AccusedPerson, ValidationResult
''')
if not rows:
    display(Markdown("_No events._"))
else:
    md = ["| When | Player | Event | Case | Accused | Result |",
          "|------|--------|-------|------|---------|--------|"]
    for r in rows:
        md.append(f"| {r['Timestamp']} | `{r['PlayerId']}` | {r['EventType']} | "
                  f"{r['CaseId']} | {r['AccusedPerson']} | {r['ValidationResult']} |")
    display(Markdown("\n".join(md)))